# 17. Hashing Interview Problems: Set 2

More classics that use **maps and sets in different ways**: sets for O(1) membership, two-way maps for bijections, meet-in-the-middle pair sums, counting by position.

**Learning order:** LONGEST CONSECUTIVE → 4SUM II → ISOMORPHIC → WORD PATTERN → LONGEST PALINDROME → FIRST UNIQUE → VALID SUDOKU → BRICK WALL → INTERVIEW THINKING

## 1. Longest Consecutive Sequence (O(n) without sorting)

Put all numbers in a **set**. Only start counting from numbers that are **the start of a sequence** (`x - 1` is not in the set). Then walk `x, x+1, x+2...` while present. Each number is visited at most twice → O(n).

In [1]:
def longest_consecutive(nums):
    s, best = set(nums), 0
    for x in s:
        if x - 1 not in s:                         # x starts a sequence
            length = 1
            while x + length in s:
                length += 1
            best = max(best, length)
    return best

print(longest_consecutive([100, 4, 200, 1, 3, 2]), longest_consecutive([0, 3, 7, 2, 5, 8, 4, 6, 0, 1]))   # 4 9

4 9


## 2. 4Sum II (meet in the middle)

Four arrays; count tuples `(i, j, k, l)` with `a[i] + b[j] + c[k] + d[l] == 0`. O(n⁴) brute force. Split into two halves: store all sums `a + b` with counts in a dict, then for every `c + d` look up `-(c + d)`. O(n²).

In [2]:
from collections import Counter

def four_sum_count(a, b, c, d):
    ab = Counter(x + y for x in a for y in b)
    return sum(ab[-(x + y)] for x in c for y in d)

print(four_sum_count([1, 2], [-2, -1], [-1, 2], [0, 2]))   # 2

2


## 3. Isomorphic Strings and Word Pattern (bijection)

Each character must map to **exactly one** character **and vice versa**. Use **two** dictionaries (or compare first-occurrence patterns).

In [3]:
def is_isomorphic(s, t):
    if len(s) != len(t):
        return False
    s_to_t, t_to_s = {}, {}
    for a, b in zip(s, t):
        if s_to_t.setdefault(a, b) != b or t_to_s.setdefault(b, a) != a:
            return False
    return True

def word_pattern(pattern, s):
    words = s.split()
    if len(pattern) != len(words):
        return False
    return is_isomorphic(pattern, words)       # same bijection idea, words instead of characters

print(is_isomorphic("egg", "add"), is_isomorphic("foo", "bar"), is_isomorphic("paper", "title"), is_isomorphic("badc", "baba"))   # True False True False
print(word_pattern("abba", "dog cat cat dog"), word_pattern("abba", "dog cat cat fish"), word_pattern("aaaa", "dog cat cat dog"))   # True False False

True False True False
True False False


## 4. Longest Palindrome from letters

Count letters. Every letter contributes its **pairs** (`count // 2 * 2`). If any letter has an **odd** count, one extra letter can go in the middle (+1).

In [4]:
def longest_palindrome(s):
    counts = Counter(s)
    length = sum(c // 2 * 2 for c in counts.values())
    return length + (1 if any(c % 2 for c in counts.values()) else 0)

print(longest_palindrome("abccccdd"), longest_palindrome("a"))   # 7 1

7 1


## 5. First Unique Character

Two passes: count, then find the first character whose count is `1`. O(n), O(26) space.

In [5]:
def first_uniq_char(s):
    counts = Counter(s)
    for i, ch in enumerate(s):
        if counts[ch] == 1:
            return i
    return -1

print(first_uniq_char("leetcode"), first_uniq_char("loveleetcode"), first_uniq_char("aabb"))   # 0 2 -1

0 2 -1


## 6. Valid Sudoku (sets per row, column, box)

Only the filled cells matter. Keep a set for each row, column and 3×3 box (`box = (r // 3, c // 3)`). If a digit is already in any of them, the board is invalid. O(81).

In [6]:
def is_valid_sudoku(board):
    rows, cols, boxes = [set() for _ in range(9)], [set() for _ in range(9)], [set() for _ in range(9)]
    for r in range(9):
        for c in range(9):
            v = board[r][c]
            if v == ".":
                continue
            b = r // 3 * 3 + c // 3
            if v in rows[r] or v in cols[c] or v in boxes[b]:
                return False
            rows[r].add(v); cols[c].add(v); boxes[b].add(v)
    return True

board = ["53..7....", "6..195...", ".98....6.", "8...6...3", "4..8.3..1", "7...2...6", ".6....28.", "...419..5", "....8..79"]
print(is_valid_sudoku([list(r) for r in board]))   # True
board[0] = "83..7...."                             # two 8s in column 0
print(is_valid_sudoku([list(r) for r in board]))   # False

True
False


## 7. Brick Wall (count by position)

Draw a vertical line crossing the **fewest bricks**. A line at a **gap** between bricks crosses none in that row. Count, for every position (prefix sum of brick widths, excluding the wall's edge), how many rows have a gap there. Answer = `rows - max gaps at one position`.

In [7]:
def least_bricks(wall):
    gaps = Counter()
    for row in wall:
        pos = 0
        for brick in row[:-1]:                     # skip the last brick: the right edge is not allowed
            pos += brick
            gaps[pos] += 1
    return len(wall) - max(gaps.values(), default=0)

print(least_bricks([[1, 2, 2, 1], [3, 1, 2], [1, 3, 2], [2, 4], [3, 1, 2], [1, 3, 1, 1]]))   # 2

2


## 8. Interview thinking

**Signals:** "consecutive", "pairs from several lists", "one-to-one mapping", "count by position", "valid board".

**Say:** "I trade O(n) space for time. A set gives O(1) membership, so the scan is O(n) instead of sorting O(n log n)." / "Two maps because the mapping must be one-to-one both ways."

**Common mistakes:** starting the consecutive count from every number (O(n²)), one-way mapping for isomorphic strings (`"badc"` / `"baba"` wrongly passes), forgetting the middle odd letter, counting the right edge in Brick Wall.

## Quick Revision

- Consecutive: set + only start from `x - 1 not in set`.
- 4Sum II: dict of `a + b`, look up `-(c + d)`.
- Bijection: two maps (or compare patterns).
- Longest palindrome: pairs + one odd.
- Valid sudoku: set per row/col/box. Brick wall: most common gap.